# NH3 / H2 Qubit Hamiltonian (Colab Minimal)

This notebook now defers all logic to the repository code. Workflow:

1. Install pinned dependencies (Qiskit 2.x, qiskit-nature, PySCF).
2. Clone the repo.
3. Run the CLI script for NH3 (active space, 6 qubits) or H2 (4 qubits).
4. (Optional) Use fallback only (`--force-precomputed`) if PySCF fails.

See repository README for details and provenance notes.


In [ ]:
# === Master Environment + Repo Setup (Run FIRST) ===
import sys, subprocess, importlib, os, pathlib, numpy as np

# 1. Pin core quantum chemistry stack (idempotent)
PKGS = ['qiskit==2.1.2','qiskit-nature==0.7.2','pyscf==2.6.1','qiskit-aer']
subprocess.check_call([sys.executable,'-m','pip','install','--upgrade','--no-cache-dir']+PKGS)

# 2. Clone / update repo containing vqeskeletal.py (GroundStateFinder)
REPO_URL = 'https://github.com/Kukyos/GroundStateFinder.git'
REPO_DIR = pathlib.Path('GroundStateFinder')
if not REPO_DIR.exists():
    subprocess.check_call(['git','clone','--depth','1',REPO_URL])
else:
    try:
        subprocess.check_call(['git','-C',str(REPO_DIR),'pull','--ff-only'])
    except Exception as e:
        print('Git pull failed (continuing):', e)

# 3. Ensure repo root and src on sys.path
paths_added = []
for p in [REPO_DIR, REPO_DIR/'src']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        paths_added.append(str(p))
print('Added to sys.path:', paths_added)

# 4. Verify presence of vqeskeletal.py
vqefile = REPO_DIR/'vqeskeletal.py'
if not vqefile.exists():
    print('FATAL: vqeskeletal.py not found in cloned repo; aborting.')
else:
    print('Found vqeskeletal.py at', vqefile)

# 5. Core chemistry imports
from pyscf import gto, scf, ao2mo
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp

# 6. Import repo module
import vqeskeletal as vsk
importlib.reload(vsk)

# 7. Version report
print('\nVersions:')
for mod_name in ['qiskit','qiskit_nature','pyscf','qiskit_aer']:
    try:
        m = importlib.import_module(mod_name)
        print(f'  {mod_name:14s}:', getattr(m,'__version__','?'))
    except Exception as e:
        print(f'  {mod_name:14s}: MISSING ({e})')

# 8. Shared constants (NH3 active space 6 qubits)
NH3_GEOM = 'N 0 0 0; H 0.9377 0 -0.3816; H -0.4688 0.8119 -0.3816; H -0.4688 -0.8119 -0.3816'
ELECTRONS_ALPHA = 2
ELECTRONS_BETA  = 2
ACTIVE_SPATIAL_ORBS = 3  # -> 6 spin orbitals

print('\nEnvironment + repository initialization complete.')

Direct NH3 6-qubit active-space build and Pauli expansion (no error handling).

In [ ]:
# (Former Cell 1) All setup moved to the Master Environment cell (Cell 2). Nothing needed here.
print('Setup already performed. Skip this cell or delete.')

## Note
Padding adds zero-coefficient Pauli strings to reach the requested minimum; physics unaffected.

In [ ]:
# === 2. Build NH3 Active-Space Pauli Hamiltonian (Working Cell 2) ===
# Deterministic active-space construction (6 qubits) from PySCF integrals – NO FALLBACKS.

# SCF
mol = gto.M(atom=NH3_GEOM, basis='sto-3g', unit='Angstrom')
mf = scf.RHF(mol).run()
print(f'SCF energy: {mf.e_tot:.8f} Hartree')

# MO integrals
C = mf.mo_coeff
h_core_ao = mf.get_hcore()
h1_mo = C.T @ h_core_ao @ C
nmo = C.shape[1]
# 2-electron integrals (chemist) in MO basis
eri_mo_full = ao2mo.restore(1, ao2mo.full(mf._eri, C), nmo)

# Active space selection (first 3 spatial orbitals)
act = list(range(ACTIVE_SPATIAL_ORBS))
h1_act = h1_mo[np.ix_(act, act)]
eri_act = eri_mo_full[np.ix_(act, act, act, act)]

# Build ElectronicEnergy from raw integrals and wrap minimal problem info
# (We rely on qiskit-nature API directly; if this errors we STOP and fix.)
ee_act = ElectronicEnergy.from_raw_integrals(h1_act, eri_act)
problem_active = ElectronicStructureProblem(ee_act)

# Minimal wrapper supplying attributes AnsatzPlugin expects (accept flexible arg names)
class MiniProblem:
    def __init__(self, num_spin_orbitals=None, n_spin=None, num_alpha=None, n_alpha=None, num_beta=None, n_beta=None):
        self.num_spin_orbitals = num_spin_orbitals if num_spin_orbitals is not None else n_spin
        a = num_alpha if num_alpha is not None else n_alpha
        b = num_beta if num_beta is not None else n_beta
        self.num_particles = (a, b)
        if self.num_spin_orbitals is None or a is None or b is None:
            raise ValueError('MiniProblem requires spin orbitals and both particle counts.')

# Instantiate with explicit keywords
mini_problem = MiniProblem(num_spin_orbitals=2*ACTIVE_SPATIAL_ORBS,
                           num_alpha=ELECTRONS_ALPHA,
                           num_beta=ELECTRONS_BETA)

# Map fermionic Hamiltonian (robust to API return shape; still NO silent fallback)
mapper = JordanWignerMapper()
raw_ops = problem_active.second_q_ops()
print(f"second_q_ops() return type: {type(raw_ops)}")
if isinstance(raw_ops, dict):
    if 'ElectronicEnergy' not in raw_ops:
        raise KeyError("'ElectronicEnergy' key missing in second_q_ops() dict; keys: " + str(list(raw_ops.keys())))
    ferm_op = raw_ops['ElectronicEnergy']
elif isinstance(raw_ops, tuple):
    # Expect (main_op, aux_ops)
    if len(raw_ops) != 2:
        raise ValueError(f"Tuple from second_q_ops() length {len(raw_ops)} != 2; inspect manually.")
    ferm_op, aux_ops = raw_ops
    print(f"Extracted main fermionic operator from tuple; aux count: {len(aux_ops) if hasattr(aux_ops,'__len__') else 'N/A'}")
elif isinstance(raw_ops, list):
    if len(raw_ops) == 0:
        raise ValueError('Empty list from second_q_ops().')
    ferm_op = raw_ops[0]
    print('Warning: second_q_ops() returned list; using first element as main operator.')
else:
    raise TypeError(f"Unhandled type from second_q_ops(): {type(raw_ops)}")

qubit_op = mapper.map(ferm_op)

# Strict sanity checks (no silent fallbacks)
expected_qubits = 2 * ACTIVE_SPATIAL_ORBS
assert qubit_op.num_qubits == expected_qubits, f"Mapped qubits {qubit_op.num_qubits} != expected {expected_qubits}" 
assert mini_problem.num_spin_orbitals == expected_qubits, "MiniProblem spin orbital mismatch"
assert sum(mini_problem.num_particles) == ELECTRONS_ALPHA + ELECTRONS_BETA, "Electron count mismatch"

# Term stats
labels = qubit_op.paulis.to_labels()
nonzero = [ (lbl, coeff) for lbl, coeff in zip(labels, qubit_op.coeffs) if abs(complex(coeff)) > 1e-12 ]
print(f"Qubits: {qubit_op.num_qubits}")
print(f"Non-zero Pauli terms: {len(nonzero)}")
print("Sample terms (first 10):")
for (lbl, coeff) in nonzero[:10]:
    print(f"  {coeff.real:+.8f} * {lbl}")

# System dictionary used downstream (NO fallbacks — this is the single source)
ham_system = {
    'problem_active': mini_problem,
    'mapper': mapper,
    'hamiltonian_active': qubit_op,
    'num_qubits': qubit_op.num_qubits,
    'basis': 'sto3g',
    'geometry': NH3_GEOM,
    'fallback': False
}
print('Hamiltonian system ready (use in later cells).')

In [ ]:
# === 3. Build UCCSD Ansatz (Working Cell 3) ===
import importlib, vqeskeletal as vsk
importlib.reload(vsk)
from vqeskeletal import AnsatzPlugin

# Build ansatz from ham_system (expects mapper + mini problem)
ansatz = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=True)
ansatz.build_from_hamiltonian(ham_system)
info = ansatz.get_ansatz_info()
print('Ansatz info:', {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})

# Store for later VQE runs
uccsd_ansatz = ansatz
print('Stored as uccsd_ansatz.')

### UCCSD Ansatz (match NH3 Hamiltonian)
Build a UCCSD circuit sized to the NH3 active-space Hamiltonian using integrated package helpers.

In [ ]:
# OPTIONAL: Compact UCCSD summary using provided helper utilities (repo already cloned)
import numpy as np, importlib
try:
    from groundstate import build_molecule_qubit_hamiltonian, uccsd_for_hamiltonian, circuit_summary
except ImportError:
    print('groundstate helpers not found (ensure repo cloned). Skipping summary.')
else:
    nh3_geom = NH3_GEOM
    ham = build_molecule_qubit_hamiltonian('NH3')
    ansatz_tmp, params_tmp = uccsd_for_hamiltonian(nh3_geom, ham, param_scale=0.02, seed=42)
    particles = ansatz_tmp.num_particles if isinstance(ansatz_tmp.num_particles,(tuple,list)) else (ansatz_tmp.num_particles, ansatz_tmp.num_particles)
    active_e = sum(particles)
    spatial = ansatz_tmp.num_spatial_orbitals
    print(f"Active space: {active_e} electrons, {spatial} orbitals -> {ansatz_tmp.num_qubits} qubits")
    print("Parameters:", np.array2string(params_tmp, separator=' ', max_line_width=120))
    print("\nCircuit (compact, high-level):")
    print(circuit_summary(ansatz_tmp, max_gates=25, decompose=False))

### VQE Skeleton Integration
Demonstrate integrating the repository's VQE skeleton (`vqeskeletal.py`) with a simple optimizer plugin.

This cell will:
1. Ensure the repo clone is present / updated.
2. Import the skeleton classes.
3. Define a minimal gradient-free optimizer (coordinate search) that fits the plugin interface.
4. Build the Hamiltonian + UCCSD ansatz via the skeleton plugins.
5. Run a mock VQE (note: expectation function is a placeholder returning 0.0 in the skeleton).

You can later replace the placeholder expectation with a real Estimator evaluation and plug in a hybrid (global→local) optimizer.

In [ ]:
# Minimal coordinate-descent optimizer demo (repo & imports already set up)
import importlib, numpy as np, vqeskeletal as vsk
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, VQE

class CoordinateDescentOptimizer(vsk.ClassicalOptimizerPlugin):
    def __init__(self, max_iters=5, step=0.2, shrink=0.5, tol=1e-6, verbose=True):
        self.max_iters=max_iters; self.step=step; self.shrink=shrink; self.tol=tol; self.verbose=verbose
    def optimize(self, objective_function, initial_params):
        params = np.array(initial_params, dtype=float)
        best_val = objective_function(params); step=self.step
        if self.verbose: print(f'Initial value: {best_val}')
        for it in range(self.max_iters):
            improved=False
            for i in range(len(params)):
                for d in (+1,-1):
                    trial = params.copy(); trial[i]+=d*step
                    val=objective_function(trial)
                    if val < best_val - self.tol:
                        best_val=val; params=trial; improved=True
                        if self.verbose: print(f'Iter {it} param {i} {"+" if d>0 else "-"} -> {best_val}')
            if not improved:
                step*=self.shrink
                if self.verbose: print('No improvement; shrink step ->', step)
                if step < self.tol:
                    if self.verbose: print('Converged (step below tol)')
                    break
        return params

ansatz_plugin = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
ham_plugin = HamiltonianPlugin()
zne_plugin = ZNEDenoiserPlugin(verbose=False)
coord_opt = CoordinateDescentOptimizer()

ham_qubit = ham_plugin.get_hamiltonian()
ansatz_plugin.build_from_hamiltonian(ham_qubit)
print('Ansatz info:', {k:v for k,v in ansatz_plugin.get_ansatz_info().items() if k in ['num_qubits','num_parameters','circuit_depth']})
init = ansatz_plugin.get_initial_parameters('random_small')
vqe_instance = VQE(ansatz_plugin, ham_plugin, coord_opt, zne_plugin)
opt_params, energy = vqe_instance.run(init)
print('Returned (placeholder / mitigated) energy:', energy)

In [ ]:
# Single energy evaluation via skeleton (estimator if available)
import vqeskeletal as vsk
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, VQE

ansatz_plugin2 = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
ham_plugin2 = HamiltonianPlugin()
ham2 = ham_plugin2.get_hamiltonian()
ansatz_plugin2.build_from_hamiltonian(ham2)

class NoOpOpt(vsk.ClassicalOptimizerPlugin):
    def optimize(self, fn, init): return init

vqe2 = VQE(ansatz_plugin2, ham_plugin2, NoOpOpt(), ZNEDenoiserPlugin(verbose=False))
params0 = ansatz_plugin2.get_initial_parameters('zero')
val = vqe2.objective_function(params0)
print('Single energy evaluation:', val)

In [ ]:
# Hybrid SPSA->COBYLA VQE run
from importlib import reload
import vqeskeletal as vsk
reload(vsk)
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, HybridSPSAThenCOBYLA, VQE

# Build plugins
ham_plugin = HamiltonianPlugin()
ansatz_plugin = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
zne_plugin = ZNEDenoiserPlugin()
optimizer_plugin = HybridSPSAThenCOBYLA(spsa_iters=20, switch_tol=5e-3, min_spsa=10, force_cobyla=True, verbose=True)

# Build ansatz first (explicit) then run VQE
ham_sys = ham_plugin.get_hamiltonian()
ansatz_plugin.build_from_hamiltonian(ham_sys)
info = ansatz_plugin.get_ansatz_info()
print('Ansatz info (hybrid run):', {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})

initial = ansatz_plugin.get_initial_parameters('random_small')
print('Initial (first 6):', initial[:6])

vqe = VQE(ansatz_plugin, ham_plugin, optimizer_plugin, zne_plugin, verbose=True)
params, energy = vqe.run(initial)
print('\nHybrid VQE complete. Energy:', energy)
print('Optimized params (first 6):', params[:6] if params is not None else None)


In [ ]:
# (Removed) Previously used for in-notebook hot patching of vqeskeletal.VQE measurement logic.
# Now obsolete because the repository file `vqeskeletal.py` contains the finalized implementation.
# Keeping this placeholder cell to avoid execution order surprises. Safe to delete if desired.


## Updated Workflow Summary
1. Build NH3 Hamiltonian manually from PySCF integrals (avoids driver basis errors).
2. Package `ham_system` with mapper + minimal problem info.
3. Construct full UCCSD + HF ansatz via `AnsatzPlugin`.
4. Run Hybrid SPSA -> COBYLA VQE on the real 62-term Hamiltonian.
5. (Optional) Extend with multi-factor ZNE or larger ansatz reps.

You can rerun only the Hamiltonian cell + the UCCSD VQE cell to test different optimizer settings without reinstalling anything.

In [ ]:
# === 4. Basic VQE (SPSA only, no ZNE) ===
from vqeskeletal import VQE, ZNEDenoiserPlugin, SPSAOptimizer

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

basic_optimizer = SPSAOptimizer(max_iter=25, a=0.25, c=0.15, tol=5e-4, verbose=True)
no_zne = ZNEDenoiserPlugin(noise_factors=[1.0], extrapolation_method='linear', verbose=False)

vqe_basic = VQE(uccsd_ansatz, DirectHam(ham_system), basic_optimizer, no_zne, verbose=True)
init_basic = uccsd_ansatz.get_initial_parameters('random_small')
params_basic, energy_basic = vqe_basic.run(init_basic)
print('\n[Basic VQE] Final energy:', energy_basic)

In [ ]:
# === 5. Hybrid VQE (SPSA -> COBYLA, no ZNE) ===
from vqeskeletal import HybridSPSAThenCOBYLA, ZNEDenoiserPlugin, VQE

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

hybrid_opt = HybridSPSAThenCOBYLA(spsa_iters=40, switch_tol=5e-3, min_spsa=12, force_cobyla=True, verbose=True)
no_zne2 = ZNEDenoiserPlugin(noise_factors=[1.0], verbose=False)

vqe_hybrid = VQE(uccsd_ansatz, DirectHam(ham_system), hybrid_opt, no_zne2, verbose=True)
init_hybrid = uccsd_ansatz.get_initial_parameters('random_small')
params_hybrid, energy_hybrid = vqe_hybrid.run(init_hybrid)
print('\n[Hybrid VQE] Final energy:', energy_hybrid)

In [ ]:
# === 6. Hybrid VQE + ZNE (Richardson) ===
from vqeskeletal import ZNEDenoiserPlugin, VQE, HybridSPSAThenCOBYLA

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

zne_plugin = ZNEDenoiserPlugin(noise_factors=[1.0, 3.0, 5.0], extrapolation_method='richardson', verbose=True)
# Reuse hybrid optimizer settings
hybrid_opt2 = HybridSPSAThenCOBYLA(spsa_iters=40, switch_tol=5e-3, min_spsa=12, force_cobyla=True, verbose=True)

vqe_hybrid_zne = VQE(uccsd_ansatz, DirectHam(ham_system), hybrid_opt2, zne_plugin, verbose=True)
init_hybrid_zne = uccsd_ansatz.get_initial_parameters('random_small')
params_hybrid_zne, energy_hybrid_zne = vqe_hybrid_zne.run(init_hybrid_zne)
print('\n[Hybrid+ZNE VQE] Final (extrapolated) energy:', energy_hybrid_zne)
print('\nZNE analysis:', zne_plugin.get_zne_analysis())

In [ ]:
# (Replaced) Old mixed fallback VQE cell removed in favor of structured cells below.
print('Placeholder: old VQE cell replaced. See new Cells 4,5,6 for runs.')